In [11]:
import numpy as np
import xarray as xr
import pandas as pd
import os

import matplotlib.pyplot as plt

%run -i wave_stats.py
%run -i parse_spc_file.py

parse_spc_file.py loaded.


### Assumes we have processed the bulk output
Load the bulk fluxes calcualted at the contour points

In [5]:
ds_tp = xr.load_dataset( "adcirc_bulk_fluxes_tp.nc" )
ds_tp

<xarray.Dataset> Size: 12MB
Dimensions:             (time: 431, point: 425)
Coordinates:
  * time                (time) datetime64[ns] 3kB 2022-09-23T12:15:00 ... 202...
  * point               (point) int64 3kB 0 1 2 3 4 5 ... 420 421 422 423 424
Data variables: (12/24)
    Hs_tp               (time, point) float32 733kB 0.0001161 0.0001428 ... 2.18
    TPS_tp              (time, point) float32 733kB 0.0 0.0 0.0 ... 8.843 8.855
    depth_p             (point) float32 2kB 31.53 31.37 30.82 ... 31.25 31.36
    lon_p               (point) float32 2kB -88.0 -87.98 ... -82.44 -82.44
    lat_p               (point) float32 2kB 29.96 29.96 29.96 ... 25.54 25.52
    x_p                 (point) float32 2kB -1.759e+05 -1.739e+05 ... 3.557e+05
    ...                  ...
    Eflux_mag_tp        (time, point) float32 733kB 0.0 0.0 0.0 ... 47.46 47.31
    Eflux_mag_wrms_tp   (time, point) float32 733kB 0.0 0.0 ... 0.2503 0.1591
    EfluxS_mag_tp       (time, point) float32 733kB 0.0 0.0 0.0 ... 1.888 1.278
    EfluxS_x_tp         (time, point) float32 733kB 0.0 0.0 0.0 ... 1.868 1.266
    EfluxS_y_tp         (time, point) float32 733kB 0.0 0.0 ... 0.2754 0.1735
    EfluxS_mag_wrms_tp  (time, point) float32 733kB 0.0 0.0 ... 0.3573 0.2778
Attributes:
    title:        Fluxes from bulk parameters at contour points
    notes:        lon/lat from ds_Hs (deg); x/y are projected to {crs_out}; d...
    conventions:  CF-1.8
    author:       Chris Sherwood
    institution:  USGS Woods Hole
    email:        csherwood@usgs.gov
    source:       ADCIRC output 63 downloaded from Design-Safe
    history:      Written 2025-09-17T14:00:08.454749+00:00 by extract_eflux_a...
    Conventions:  CF-1.8

In [13]:
import xarray as xr
import os

# Directory with your .spc files
data_dir = "F:/crs/proj/2025_NOPP_comparison/helene_adcirc_model_results/spec_files/"

# Number of contour points from ds_tp
npts = ds_tp.dims["point"]

all_specs = []

for pid in range(npts):
    fname = os.path.join(data_dir, f"bnd{pid+1}.spc")  # files start at 1
    if not os.path.exists(fname):
        print(f"Missing file: {fname}")
        continue

    depth = float(ds_tp["depth_p"].isel(point=pid).values)
    nx = float(ds_tp["nx"].isel(point=pid).values)  # assuming you added nx, ny to ds_tp
    ny = float(ds_tp["ny"].isel(point=pid).values)

    ds_spec = read_swan_spec(fname, h=depth, normx=nx, normy=ny)

    # attach a point dimension
    ds_spec = ds_spec.expand_dims(point=[pid])

    # double-check lat/lon match (optional)
    lat_ref = float(ds_tp["lat_p"].isel(point=pid).values)
    lon_ref = float(ds_tp["lon_p"].isel(point=pid).values)
    if abs(lat_ref - float(ds_spec["lat"].values)) > 1e-3 or \
       abs(lon_ref - float(ds_spec["lon"].values)) > 1e-3:
        print(f"⚠️ Point {pid}: lat/lon mismatch (mesh {lat_ref:.3f},{lon_ref:.3f} "
              f"vs spc {float(ds_spec['lat'].values):.3f},{float(ds_spec['lon'].values):.3f})")

    all_specs.append(ds_spec)

# Merge along point dimension
if all_specs:
    master_ds = xr.concat(all_specs, dim="point")
else:
    master_ds = None

print(master_ds)


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_48012\3779486170.py:8: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  npts = ds_tp.dims["point"]
C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_48012\3779486170.py:39: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  master_ds = xr.concat(all_specs, dim="point")


<xarray.Dataset> Size: 3GB
Dimensions:        (point: 425, time: 145, freq: 41, dir: 36)
Coordinates:
  * point          (point) int64 3kB 0 1 2 3 4 5 6 ... 419 420 421 422 423 424
  * time           (time) datetime64[ns] 1kB 2024-09-25 ... 2024-09-28
  * freq           (freq) float64 328B 0.0314 0.0345 0.038 ... 1.174 1.291 1.42
  * dir            (dir) float64 288B 5.0 15.0 25.0 35.0 ... 335.0 345.0 355.0
    lon            (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44
    lat            (point) float64 3kB 29.96 29.96 29.96 ... 25.56 25.54 25.52
Data variables: (12/24)
    S2d            (point, time, freq, dir) float32 364MB 0.0 0.0 ... 0.0 0.0
    Ef             (point, time, freq) float32 10MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    Dth            (point, time, dir) float32 9MB 8.291e-10 ... 3.278e-05
    m0             (point, time) float32 246kB 0.0001106 0.0001108 ... 0.004758
    Hm0            (point, time) float32 246kB 0.04207 0.04211 ... 0.2812 0.2759
    m1          

In [14]:
master_ds

<xarray.Dataset> Size: 3GB
Dimensions:        (point: 425, time: 145, freq: 41, dir: 36)
Coordinates:
  * point          (point) int64 3kB 0 1 2 3 4 5 6 ... 419 420 421 422 423 424
  * time           (time) datetime64[ns] 1kB 2024-09-25 ... 2024-09-28
  * freq           (freq) float64 328B 0.0314 0.0345 0.038 ... 1.174 1.291 1.42
  * dir            (dir) float64 288B 5.0 15.0 25.0 35.0 ... 335.0 345.0 355.0
    lon            (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44
    lat            (point) float64 3kB 29.96 29.96 29.96 ... 25.56 25.54 25.52
Data variables: (12/24)
    S2d            (point, time, freq, dir) float32 364MB 0.0 0.0 ... 0.0 0.0
    Ef             (point, time, freq) float32 10MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    Dth            (point, time, dir) float32 9MB 8.291e-10 ... 3.278e-05
    m0             (point, time) float32 246kB 0.0001106 0.0001108 ... 0.004758
    Hm0            (point, time) float32 246kB 0.04207 0.04211 ... 0.2812 0.2759
    m1             (point, time) float64 493kB 5e-05 4.975e-05 ... 0.0007375
    ...             ...
    Dir_flux_R     (point, time) float32 246kB 0.8268 0.8315 ... 0.79 0.7902
    EfluxS_2d      (point, time, freq, dir) float64 728MB 0.0 0.0 ... 0.0 0.0
    EfluxS_2d_pos  (point, time, freq, dir) float64 728MB 0.0 0.0 ... 0.0 0.0
    EfluxS_f       (point, time, freq) float64 20MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    EfluxS_dir     (point, time, dir) float64 18MB 6.856e-06 6.315e-05 ... 15.15
    EfluxS_total   (point, time) float64 493kB 0.1395 0.1434 ... 29.69 28.31
Attributes:
    source:     SWAN .spec reader (QUANT once, repeated date-and-time blocks;...
    units_S2d:  m2/Hz/degr
    quantity:   VaDens